<a href="https://colab.research.google.com/github/Shadid-dev/INSE6180-cluster-anonymization/blob/main/INSE6180_cluster_anonymization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/Shadid-dev/INSE6180-cluster-anonymization/main/data/"
df = pd.read_csv(URL + "adult.csv")

print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [2]:
QIS = ["age", "sex", "race", "marital-status", "education",
       "native-country", "workclass", "occupation"]
TARGET = "income"

df = df.rename(columns={"gender": "sex"})
df = df[QIS + [TARGET]]
df = df.replace("?", "Unknown")

print(df.shape)
print((df == "Unknown").sum())

(48842, 9)
age                  0
sex                  0
race                 0
marital-status       0
education            0
native-country     857
workclass         2799
occupation        2809
income               0
dtype: int64


In [3]:
group_size = df.groupby(QIS)[TARGET].transform("size")
unique = (group_size == 1).sum()
print("Unique on all 8 QIs:", unique, f"({unique/len(df):.1%})")

Unique on all 8 QIs: 20593 (42.2%)


In [4]:
H = {}
for q in QIS:
    h = pd.read_csv(URL + f"adult_hierarchy_{q}.csv", sep=";", header=None, dtype=str)
    if q in ["workclass", "occupation", "native-country"]:
        h.loc[len(h)] = ["Unknown", "Unknown"] + ["*"] * (h.shape[1] - 2)
    H[q] = h

for q in QIS:
    print(q, "-> top rung =", H[q].shape[1] - 1)
H["age"].head(3)

age -> top rung = 4
sex -> top rung = 1
race -> top rung = 1
marital-status -> top rung = 2
education -> top rung = 3
native-country -> top rung = 2
workclass -> top rung = 2
occupation -> top rung = 2


,0,1,2,3,4
0,1,0-4,0-9,0-19,*
1,2,0-4,0-9,0-19,*
2,3,0-4,0-9,0-19,*


In [5]:
def generalize(column, q, level):
    ladder = dict(zip(H[q][0], H[q][level]))
    return column.astype(str).map(ladder)

for level in range(5):
    t = df.copy()
    t["age"] = generalize(df["age"], "age", level)
    size = t.groupby(QIS)[TARGET].transform("size")
    print("age rung", level, "| example:", t["age"].iloc[0],
          "| unique:", (size == 1).sum(), f"({(size == 1).mean():.1%})")

age rung 0 | example: 25 | unique: 20593 (42.2%)
age rung 1 | example: 20-24 | unique: 12401 (25.4%)
age rung 2 | example: 20-29 | unique: 9901 (20.3%)
age rung 3 | example: 20-39 | unique: 7939 (16.3%)
age rung 4 | example: * | unique: 5539 (11.3%)


In [6]:
import itertools

K = 5            # crowd size required
MAX_SUPP = 0.05  # at most 5% of rows may be deleted

top = {q: H[q].shape[1] - 1 for q in QIS}
G = {q: [generalize(df[q], q, l) for l in range(top[q] + 1)] for q in QIS}

def loss(levels):
    return sum(levels[i] / top[q] for i, q in enumerate(QIS)) / len(QIS)

options = list(itertools.product(*[range(top[q] + 1) for q in QIS]))
options.sort(key=loss)
print("Combinations to try:", len(options))

for tried, levels in enumerate(options, 1):
    t = pd.DataFrame({q: G[q][l] for q, l in zip(QIS, levels)})
    size = t.groupby(QIS)[QIS[0]].transform("size")
    deleted = (size < K).mean()
    if deleted <= MAX_SUPP:
        break

print("Tried:", tried)
print("Best rungs:", dict(zip(QIS, levels)))
print(f"Loss: {loss(levels):.3f}   Deleted: {deleted:.2%}")

Combinations to try: 6480
Tried: 1143
Best rungs: {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 1, 'native-country': 1, 'workclass': 0, 'occupation': 2}
Loss: 0.354   Deleted: 4.06%


In [8]:
!pip install -q --no-deps pycanon
from pycanon import anonymity

best = dict(zip(QIS, levels))
t = pd.DataFrame({q: G[q][best[q]] for q in QIS})
t[TARGET] = df[TARGET]

size = t.groupby(QIS)[TARGET].transform("size")
release = t[size >= K].reset_index(drop=True)

print("Released rows:", len(release), "of", len(df))
print("Smallest crowd:", release.groupby(QIS).size().min())
print("Distinct QI combinations:", release[QIS].drop_duplicates().shape[0])
print("pycanon k =", anonymity.k_anonymity(release, QIS))
release.head(3)

Released rows: 46861 of 48842
Smallest crowd: 5
Distinct QI combinations: 554
pycanon k = 5


,age,sex,race,marital-status,education,native-country,workclass,occupation,income
0,*,Male,Black,Never-married,High School,North America,Private,*,<=50K
1,*,Male,White,Married-civ-spouse,High School,North America,Private,*,<=50K
2,*,Male,White,Married-civ-spouse,Professional Education,North America,Local-gov,*,>50K


Piece &&	What it does
keep = size >= K	size comes from cell 7. This picks the same 46,861 people who are in release, so we compare like with like.
orig / anon	The same people, written two ways: exact values vs blurred values.
pd.get_dummies(...)	One-hot encoding.
n_clusters=4	Asks for 4 groups. We'll try other numbers later.
n_init=10	Runs k-means 10 times from different random starts and keeps the best result. This reduces bad luck.
random_state=0	Fixes the randomness, so you get the same answer every run.
summary	Describes each cluster in plain terms.

In [10]:
from sklearn.cluster import KMeans

keep = size >= K
orig = df.loc[keep, QIS].astype(str).reset_index(drop=True)
anon = release[QIS]

X_orig = pd.get_dummies(orig).astype(float)
X_anon = pd.get_dummies(anon).astype(float)
print("Original:", X_orig.shape, "   Anonymized:", X_anon.shape)

labels_orig = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X_orig)

o = orig.copy()
o["cluster"] = labels_orig
summary = pd.DataFrame({
    "people":    o.groupby("cluster").size(),
    "avg_age":   o.groupby("cluster")["age"].apply(lambda a: a.astype(int).mean()).round(1),
    "female_%":  o.groupby("cluster")["sex"].apply(lambda a: (a == "Female").mean() * 100).round(0),
    "married_%": o.groupby("cluster")["marital-status"].apply(lambda a: (a == "Married-civ-spouse").mean() * 100).round(0),
    "top_job":   o.groupby("cluster")["occupation"].agg(lambda a: a.value_counts().index[0]),
})
summary

Original: (46861, 169)    Anonymized: (46861, 35)


,people,avg_age,female_%,married_%,top_job
cluster,,,,,
0,8660,28.1,0.0,0.0,Craft-repair
1,7775,47.4,1.0,88.0,Exec-managerial
2,15136,36.6,100.0,14.0,Adm-clerical
3,15290,41.8,0.0,84.0,Craft-repair


In [11]:
from sklearn.metrics import adjusted_rand_score

labels_anon = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X_anon)
labels_orig2 = KMeans(n_clusters=4, n_init=10, random_state=1).fit_predict(X_orig)

print("ARI original vs anonymized:  ", round(adjusted_rand_score(labels_orig, labels_anon), 3))
print("ARI original vs original (2nd run):", round(adjusted_rand_score(labels_orig, labels_orig2), 3))

ARI original vs anonymized:   0.687
ARI original vs original (2nd run): 0.661


0.687 is not lower than 0.661, so with 4 groups , anonymization didn't damage the cluster structure


In [17]:
rows = []
for c in range(2, 9):                 # number of groups: 2, 3, ..., 8
    for s in range(5):                # 5 repeats with different random starts
        lo  = KMeans(c, n_init=10, random_state=s).fit_predict(X_orig)
        lo2 = KMeans(c, n_init=10, random_state=s + 100).fit_predict(X_orig)
        la  = KMeans(c, n_init=10, random_state=s + 100).fit_predict(X_anon)   # FIXED
        rows.append({"groups": c, "orig_vs_anon": adjusted_rand_score(lo, la),
                     "orig_vs_orig": adjusted_rand_score(lo, lo2)})

ari = pd.DataFrame(rows).groupby("groups").mean().round(3)
ari

,orig_vs_anon,orig_vs_orig
groups,,
2,0.880,1.000
3,0.598,1.000
4,0.710,0.729
5,0.601,0.677
6,0.487,0.784
7,0.455,0.678
8,0.412,0.674


After fixing the random-start bias: 2 groups is only slightly below baseline (0.88 vs 1.00), 4 groups is close (0.71 vs 0.73), and 3 and 5–8 groups are well below. Anonymization changes the cluster structure from 3 groups on.

Classification


In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

y = (df.loc[keep, TARGET].reset_index(drop=True) == ">50K").astype(int)
train, test = train_test_split(range(len(y)), test_size=0.3, random_state=0, stratify=y)

for name, X in [("original", X_orig), ("anonymized", X_anon)]:
    model = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
    model.fit(X.iloc[train], y.iloc[train])
    pred = model.predict(X.iloc[test])
    print(f"{name:11s} accuracy {accuracy_score(y.iloc[test], pred):.3f}   F1 {f1_score(y.iloc[test], pred):.3f}")

print("always-guess-<=50K accuracy:", round(1 - y.mean(), 3))

original    accuracy 0.817   F1 0.597
anonymized  accuracy 0.812   F1 0.608
always-guess-<=50K accuracy: 0.759


In [18]:
import numpy as np

def anonymize(data, qis, k=5, max_supp=0.05):
    top = {q: H[q].shape[1] - 1 for q in qis}
    G = {q: [generalize(data[q], q, l) for l in range(top[q] + 1)] for q in qis}
    options = list(itertools.product(*[range(top[q] + 1) for q in qis]))
    options.sort(key=lambda lv: sum(lv[i] / top[q] for i, q in enumerate(qis)) / len(qis))
    for levels in options:
        t = pd.DataFrame({q: G[q][l] for q, l in zip(qis, levels)})
        size = t.groupby(qis)[qis[0]].transform("size")
        if (size < k).mean() <= max_supp:
            keep = (size >= k).values
            return t[keep].reset_index(drop=True), keep, dict(zip(qis, levels))

def run_one(data, qis, k=5, seeds=3, groups=range(2, 9)):
    data = data.reset_index(drop=True)
    anon, keep, levels = anonymize(data, qis, k)
    orig = data.loc[keep, qis].astype(str).reset_index(drop=True)
    Xo = pd.get_dummies(orig).astype(float)
    Xa = pd.get_dummies(anon).astype(float)

    ari_anon, ari_base = [], []
    for c in groups:
        for s in range(seeds):
            lo  = KMeans(c, n_init=10, random_state=s).fit_predict(Xo)
            lo2 = KMeans(c, n_init=10, random_state=s + 100).fit_predict(Xo)
            la  = KMeans(c, n_init=10, random_state=s + 100).fit_predict(Xa)   # FIXED
            ari_anon.append(adjusted_rand_score(lo, la))
            ari_base.append(adjusted_rand_score(lo, lo2))

    y = (data.loc[keep, TARGET].reset_index(drop=True) == ">50K").astype(int)
    tr, te = train_test_split(range(len(y)), test_size=0.3, random_state=0, stratify=y)
    acc = {}
    for name, X in [("orig", Xo), ("anon", Xa)]:
        m = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
        m.fit(X.iloc[tr], y.iloc[tr])
        acc[name] = accuracy_score(y.iloc[te], m.predict(X.iloc[te]))

    return {"n": len(data), "QIs": len(qis), "deleted_%": 100 * (1 - keep.mean()),
            "ARI_anon": np.mean(ari_anon), "ARI_baseline": np.mean(ari_base),
            "ARI_gap": np.mean(ari_base) - np.mean(ari_anon),
            "acc_orig": acc["orig"], "acc_anon": acc["anon"],
            "acc_drop": acc["orig"] - acc["anon"], "levels": levels}

In [20]:
exp1 = []
for n in [1500, 5000, 15000, len(df)]:
    reps = 3 if n < len(df) else 1
    for rep in range(reps):
        sample = df.sample(n=n, random_state=rep) if n < len(df) else df
        r = run_one(sample, QIS)
        r["rep"] = rep
        exp1.append(r)
        print("done: n =", n, "rep", rep)

exp1 = pd.DataFrame(exp1)
exp1.groupby("n")[["deleted_%", "ARI_anon", "ARI_baseline", "ARI_gap",
                   "acc_orig", "acc_anon", "acc_drop"]].mean().round(3)

done: n = 1500 rep 0
done: n = 1500 rep 1
done: n = 1500 rep 2
done: n = 5000 rep 0
done: n = 5000 rep 1
done: n = 5000 rep 2
done: n = 15000 rep 0
done: n = 15000 rep 1
done: n = 15000 rep 2
done: n = 48842 rep 0


,deleted_%,ARI_anon,ARI_baseline,ARI_gap,acc_orig,acc_anon,acc_drop
n,,,,,,,
1500,4.422,0.441,0.758,0.317,0.814,0.773,0.041
5000,4.607,0.700,0.788,0.088,0.810,0.758,0.052
15000,4.109,0.622,0.771,0.149,0.815,0.783,0.032
48842,4.056,0.610,0.764,0.154,0.817,0.812,0.005


In [21]:
exp1[["n", "rep", "ARI_anon", "ARI_baseline", "ARI_gap", "acc_drop", "levels"]].round(3)

,n,rep,ARI_anon,ARI_baseline,ARI_gap,acc_drop,levels
0,1500,0,0.475,0.734,0.259,0.018,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
1,1500,1,0.406,0.758,0.352,0.061,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
2,1500,2,0.442,0.781,0.339,0.044,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
3,5000,0,0.738,0.788,0.050,0.036,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
4,5000,1,0.722,0.788,0.067,0.060,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
5,5000,2,0.641,0.789,0.147,0.060,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
6,15000,0,0.707,0.774,0.067,0.044,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
7,15000,1,0.687,0.731,0.044,0.046,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
8,15000,2,0.471,0.807,0.336,0.006,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."
9,48842,0,0.610,0.764,0.154,0.005,"{'age': 4, 'sex': 0, 'race': 0, 'marital-statu..."


In [22]:
for _, r in exp1.iterrows():
    print(r["n"], "rep", r["rep"], "| gap", round(r["ARI_gap"], 3), "|", r["levels"])

1500 rep 0 | gap 0.259 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 1, 'education': 1, 'native-country': 2, 'workclass': 2, 'occupation': 2}
1500 rep 1 | gap 0.352 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 1, 'education': 3, 'native-country': 2, 'workclass': 1, 'occupation': 1}
1500 rep 2 | gap 0.339 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 1, 'education': 1, 'native-country': 2, 'workclass': 2, 'occupation': 2}
5000 rep 0 | gap 0.05 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 3, 'native-country': 1, 'workclass': 1, 'occupation': 2}
5000 rep 1 | gap 0.067 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 3, 'native-country': 2, 'workclass': 0, 'occupation': 2}
5000 rep 2 | gap 0.147 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 3, 'native-country': 1, 'workclass': 1, 'occupation': 2}
15000 rep 0 | gap 0.067 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 3, 'native-country': 

the 2022 paper expected that more QIs means more damage, because every extra QI is something an attacker could use, so more has to be blurred. We keep the full 48,842 people and use the first 2, 4, 6, then 8 QIs.

In [23]:
exp2 = []
for m in [2, 4, 6, 8]:
    r = run_one(df, QIS[:m])
    exp2.append(r)
    print("done:", m, "QIs ->", QIS[:m])

exp2 = pd.DataFrame(exp2)
for _, r in exp2.iterrows():
    print(r["QIs"], "QIs | gap", round(r["ARI_gap"], 3), "|", r["levels"])

exp2[["QIs", "deleted_%", "ARI_anon", "ARI_baseline", "ARI_gap",
      "acc_orig", "acc_anon", "acc_drop"]].round(3)

done: 2 QIs -> ['age', 'sex']
done: 4 QIs -> ['age', 'sex', 'race', 'marital-status']
done: 6 QIs -> ['age', 'sex', 'race', 'marital-status', 'education', 'native-country']
done: 8 QIs -> ['age', 'sex', 'race', 'marital-status', 'education', 'native-country', 'workclass', 'occupation']
2 QIs | gap 0.0 | {'age': 0, 'sex': 0}
4 QIs | gap 0.0 | {'age': 0, 'sex': 0, 'race': 0, 'marital-status': 0}
6 QIs | gap 0.094 | {'age': 2, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 1, 'native-country': 1}
8 QIs | gap 0.154 | {'age': 4, 'sex': 0, 'race': 0, 'marital-status': 0, 'education': 1, 'native-country': 1, 'workclass': 0, 'occupation': 2}


,QIs,deleted_%,ARI_anon,ARI_baseline,ARI_gap,acc_orig,acc_anon,acc_drop
0,2,0.037,0.720,0.720,0.000,0.760,0.760,0.000
1,4,4.171,0.962,0.962,0.000,0.771,0.771,0.000
2,6,4.408,0.632,0.726,0.094,0.812,0.815,-0.003
3,8,4.056,0.610,0.764,0.154,0.817,0.812,0.005
